# Cross-encoder reranker (stage 1.5)

```
candidate_pairs (~7.5 / S1)  ──►  cross-encoder( "S1 name | address"  [SEP]  "candidate name | address" )  ──►  ce_logit
                                                                                                         │
                                               entity_matcher.ipynb (LightGBM uses ce_logit as a feature) ◄┘
```

The stage-1 bi-encoder embeds each record **separately**. A cross-encoder reads **both records together**, so every
token of one record can attend to the other. It catches transliteration (राम ↔ Ram), reordered or abbreviated
addresses, and chain branches that differ only in the street.

**Leakage control.** The matcher is trained on validation pairs, so validation CE scores must come from a model that
never saw those pairs. Validation S1s are split into `n_folds` folds. Fold *k*'s model is trained on the other folds
and scores fold *k* (out-of-fold). Test pairs are scored by every fold model and the logits are averaged.

**Inputs** (Kaggle → Add Data):
1. the competition dataset;
2. the output of the candidate-generation notebook (`val_candidate_pairs.parquet`, `val_queries.parquet`,
   `test_candidate_scores.parquet`). If its `embedding_model/` folder is attached too, the cross-encoder starts from
   the fine-tuned e5 weights; otherwise it starts from the base model.

**Accelerator: GPU T4 x2** (both GPUs are used). Internet must be ON if the base model has to be downloaded.

**Runtime (full):** roughly 2 × ~15–25 min training, plus ~35–60 min per fold model to score ~16M test pairs.
The first fold model's test scores are saved before the second starts. If time runs short
(`max_total_minutes`), the notebook stops after one model and the output is still valid.

In [ ]:
import os, re, csv, gc, json, time, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

IS_KAGGLE = os.path.isdir("/kaggle/working")
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else "./working")
OUT_DIR = WORK_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = ["/kaggle/input", os.getcwd()]

def find_files(name):
    hits = []
    for root in SEARCH_ROOTS:
        if os.path.isdir(root):
            for dirpath, dirnames, filenames in os.walk(root):
                dirnames[:] = [d for d in dirnames if not d.startswith(".")]
                if name in filenames:
                    hits.append(os.path.join(dirpath, name))
    return sorted(set(hits))

def find_data_root(explicit=None):
    if explicit and os.path.isfile(os.path.join(explicit, "train", "train_source1.tsv")):
        return explicit
    hits = [h for h in find_files("train_source1.tsv") if os.path.basename(os.path.dirname(h)) == "train"]
    assert hits, "competition data (train/train_source1.tsv) not found - add the dataset or set DATA_ROOT"
    return os.path.dirname(os.path.dirname(hits[0]))

def test_path(data_root, i):
    for p in (os.path.join(data_root, "test", f"test_source{i}.tsv"), os.path.join(data_root, f"test_source{i}.tsv")):
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f"test_source{i}.tsv")

def read_tsv(path, cols):
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False,
                     quoting=csv.QUOTE_NONE, usecols=cols)
    for c in cols:
        df[c] = df[c].astype(str).str.strip()
    return df[cols]

_PUNCT = r"[!-/:-@\[-`{-~¡-¿‐-‧‰-⁞　-〿।॥]"

def normalize_series(s):
    s = pd.Series(s).fillna("").astype(str)
    s = s.str.normalize("NFKC").str.lower()
    s = s.str.replace(_PUNCT, " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip().reset_index(drop=True)

def validate_id_list_file(path, header, required_ids, valid_s2=None, valid_s3=None):
    errors, seen, n_ids, n_empty = [], set(), 0, 0
    def err(m):
        if len(errors) < 30: errors.append(m)
    with open(path, encoding="utf-8") as f:
        h = f.readline().rstrip("\n").split("\t")
        if h != header: err(f"bad header {h}")
        for ln, line in enumerate(f, 2):
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 2: err(f"line {ln}: {len(parts)} columns"); continue
            s1id, ids = parts
            if s1id in seen: err(f"duplicate row {s1id}")
            seen.add(s1id)
            if s1id not in required_ids: err(f"unexpected S1 {s1id}")
            if ids == "": n_empty += 1; continue
            lst = ids.split(","); n_ids += len(lst)
            if len(set(lst)) != len(lst): err(f"{s1id}: duplicate ids")
            for x in lst:
                ok = (x.startswith("S2-") and (valid_s2 is None or x in valid_s2)) or \
                     (x.startswith("S3-") and (valid_s3 is None or x in valid_s3))
                if not ok: err(f"{s1id}: invalid id {x}")
    missing = required_ids - seen
    if missing: err(f"{len(missing):,} S1 rows missing")
    print(f"{os.path.basename(path)}: rows={len(seen):,} ids={n_ids:,} empty={n_empty:,} -> "
          + ("PASS" if not errors else f"FAIL ({len(errors)} issues)"))
    for e in errors[:10]: print("   -", e)
    return not errors

def write_id_list_file(path, header, s1_ids, groups):
    """groups: dict s1_id -> comma-joined string."""
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write("\t".join(header) + "\n")
        buf = []
        for s in s1_ids:
            buf.append(f"{s}\t{groups.get(s, '')}\n")
            if len(buf) >= 200_000: f.writelines(buf); buf = []
        f.writelines(buf)

In [ ]:
SEED = 42
DATA_ROOT = None
DEBUG_MODE = False            # True: small subsample, outputs named *_debug.parquet (ignored by the matcher)

CONFIG = {
    "base_model": "intfloat/multilingual-e5-small",   # MIT, 118M params
    "init_model": "auto",       # "auto": stage-1 fine-tuned embedding_model/ if attached, else base_model; or a path
    "max_len": 128,             # tokens for the PAIR (both records together)
    "n_folds": 2,
    "train_pairs_cap": 4_000_000 if not DEBUG_MODE else 100_000,   # per fold
    "epochs": 1,
    "batch_size": 128,
    "lr": 3e-5,
    "weight_decay": 0.01,
    "warmup_frac": 0.06,
    "max_train_minutes_per_fold": 45 if not DEBUG_MODE else 5,
    "infer_batch_size": 1024,
    "token_chunk": 500_000,     # pairs tokenised at once
    "max_total_minutes": 690,   # Kaggle hard limit is 720; stop adding fold models to test scoring past this
    "debug_val_s1": 20_000,
    "debug_test_pairs": 300_000,
}
T0 = time.time()
minutes = lambda: (time.time() - T0) / 60

import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "enable a GPU accelerator (T4 x2)"
DEV = torch.device("cuda")
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

DATA_ROOT = find_data_root(DATA_ROOT)
hits = find_files("val_candidate_pairs.parquet")
assert hits, "val_candidate_pairs.parquet not found: attach the candidate-notebook output"
RUN_DIR = os.path.dirname(hits[0])
print("data:", DATA_ROOT, "\nstage-1 run:", RUN_DIR)

def resolve_init():
    if CONFIG["init_model"] not in (None, "auto"):
        return CONFIG["init_model"]
    for info in find_files("encoder_info.json"):
        d = os.path.dirname(info)
        try:
            base = json.load(open(info)).get("base_model", "")
        except Exception:
            continue
        if base == CONFIG["base_model"] and os.path.isfile(os.path.join(d, "config.json")):
            return d
    return CONFIG["base_model"]
INIT = resolve_init()
print("cross-encoder init:", INIT)
TOK = AutoTokenizer.from_pretrained(INIT if os.path.isdir(str(INIT)) else CONFIG["base_model"])
PAD = TOK.pad_token_id
SUFFIX = "_debug" if DEBUG_MODE else ""

## Model, tokenisation and scoring helpers

In [ ]:
class CrossEncoder(nn.Module):
    """Transformer over the concatenated pair -> mean pooling -> 1 logit. autocast lives inside forward so it also
    applies in DataParallel replica threads."""
    def __init__(self, path):
        super().__init__()
        try:
            self.enc = AutoModel.from_pretrained(path)
        except Exception as e:
            print(f"could not load {path} ({e}); falling back to {CONFIG['base_model']}")
            self.enc = AutoModel.from_pretrained(CONFIG["base_model"])
        self.head = nn.Linear(self.enc.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        with torch.autocast("cuda", dtype=torch.float16):
            h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return self.head(pooled.float()).squeeze(-1)

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

def record_text(name, address):
    n = pd.Series(name).fillna("").astype(str).str.strip()
    a = pd.Series(address).fillna("").astype(str).str.strip()
    return np.where(a.to_numpy() == "", n.to_numpy(), (n + " | " + a).to_numpy()).astype(object)

def tokenize(a, b):
    return TOK(list(a), list(b), truncation="longest_first", max_length=CONFIG["max_len"], padding=False,
               return_attention_mask=False, return_token_type_ids=False)["input_ids"]

def collate(seqs):
    L = max(map(len, seqs))
    x = np.full((len(seqs), L), PAD, dtype=np.int64)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = s
    ids = torch.from_numpy(x)
    return ids.to(DEV, non_blocking=True), (ids != PAD).long().to(DEV, non_blocking=True)

@torch.no_grad()
def score_pairs(model, A, B, desc="score"):
    """A, B: object arrays of texts. Returns float32 logits. Length-sorted within each chunk for speed."""
    model.eval()
    out = np.empty(len(A), dtype=np.float32)
    bs, ch = CONFIG["infer_batch_size"], CONFIG["token_chunk"]
    pbar = tqdm(total=len(A), desc=desc, mininterval=30)
    for s in range(0, len(A), ch):
        seqs = tokenize(A[s:s + ch], B[s:s + ch])
        order = np.argsort(np.fromiter(map(len, seqs), np.int32, len(seqs)), kind="stable")
        for j in range(0, len(order), bs):
            idx = order[j:j + bs]
            x, m = collate([seqs[k] for k in idx])
            out[s + idx] = model(x, m).float().cpu().numpy()
        pbar.update(len(seqs))
        del seqs
    pbar.close()
    return out

def train_fold(A, B, y, fold):
    rng = np.random.default_rng(SEED + fold)
    order = rng.permutation(len(A))[:CONFIG["train_pairs_cap"]]
    bs = CONFIG["batch_size"]
    steps = int(np.ceil(len(order) / bs)) * CONFIG["epochs"]
    model = CrossEncoder(INIT).to(DEV)
    if N_GPU > 1:
        model = nn.DataParallel(model)
    no_decay = ("bias", "LayerNorm.weight", "layer_norm", "norm")
    params = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
               "weight_decay": CONFIG["weight_decay"]},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=CONFIG["lr"])
    sched = get_linear_schedule_with_warmup(opt, int(CONFIG["warmup_frac"] * steps), steps)
    scaler = torch.cuda.amp.GradScaler()
    lossf = nn.BCEWithLogitsLoss()
    t, step, run_loss, stop = time.time(), 0, 0.0, False
    model.train()
    pbar = tqdm(total=steps, desc=f"train fold {fold}", mininterval=30)
    for ep in range(CONFIG["epochs"]):
        ep_order = order if ep == 0 else rng.permutation(order)
        for s in range(0, len(ep_order), CONFIG["token_chunk"]):
            idx = ep_order[s:s + CONFIG["token_chunk"]]
            seqs = tokenize(A[idx], B[idx]); yy = y[idx]
            for j in range(0, len(idx), bs):
                x, m = collate(seqs[j:j + bs])
                target = torch.from_numpy(yy[j:j + bs].astype(np.float32)).to(DEV)
                loss = lossf(model(x, m), target)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                step += 1; run_loss = 0.98 * run_loss + 0.02 * loss.item() if step > 1 else loss.item()
                pbar.update(1)
                if step % 500 == 0:
                    pbar.set_postfix(loss=f"{run_loss:.4f}")
                if (time.time() - t) / 60 > CONFIG["max_train_minutes_per_fold"]:
                    stop = True; break
            del seqs
            if stop: break
        if stop: break
    pbar.close()
    print(f"fold {fold}: {step:,}/{steps:,} steps ({step * bs:,} pairs) in {(time.time() - t) / 60:.1f} min, "
          f"loss {run_loss:.4f}" + ("  [time budget hit]" if stop else ""))
    return model

## Load validation candidates and assign folds by S1

In [ ]:
vp = pd.read_parquet(os.path.join(RUN_DIR, "val_candidate_pairs.parquet"),
                     columns=["source1_entity_id", "candidate_entity_id", "label", "cosine", "prune_score",
                              "s1_name", "s1_address", "cand_name", "cand_address"])
if DEBUG_MODE:
    keep = pd.Series(vp["source1_entity_id"].unique()).sample(
        min(CONFIG["debug_val_s1"], vp["source1_entity_id"].nunique()), random_state=SEED)
    vp = vp[vp["source1_entity_id"].isin(set(keep))].reset_index(drop=True)
VA = record_text(vp["s1_name"], vp["s1_address"])
VB = record_text(vp["cand_name"], vp["cand_address"])
VY = vp["label"].to_numpy(bool)
uq = vp["source1_entity_id"].unique()
q_fold = pd.Series(np.random.default_rng(SEED).integers(0, CONFIG["n_folds"], len(uq)), index=uq)
FOLD = q_fold.reindex(vp["source1_entity_id"]).to_numpy()
print(f"validation pairs {len(vp):,} | S1 {len(uq):,} | positives {VY.mean():.2%} | "
      f"fold sizes {np.bincount(FOLD).tolist()}")
lens = np.fromiter(map(len, tokenize(VA[:20000], VB[:20000])), np.int32)
print(f"pair token length (sample): median {np.median(lens):.0f}, p95 {np.percentile(lens, 95):.0f}, "
      f"truncated at {CONFIG['max_len']}: {(lens >= CONFIG['max_len']).mean():.2%}")

## Train one model per fold → out-of-fold validation scores

In [ ]:
OOF = np.full(len(vp), np.nan, dtype=np.float32)
MODEL_PATHS = []
for k in range(CONFIG["n_folds"]):
    tr, te = FOLD != k, FOLD == k
    model = train_fold(VA[tr], VB[tr], VY[tr], k)
    OOF[te] = score_pairs(model, VA[te], VB[te], desc=f"oof fold {k}")
    print(f"fold {k} OOF AUC {roc_auc_score(VY[te], OOF[te]):.4f} | AP {average_precision_score(VY[te], OOF[te]):.4f}")
    path = Path("/tmp") / f"ce_fold{k}.pt"
    torch.save({k2: v.half() for k2, v in unwrap(model).state_dict().items()}, path)
    MODEL_PATHS.append(path)
    del model; gc.collect(); torch.cuda.empty_cache()

base = {"cross-encoder (OOF)": OOF, "stage-1 cosine": vp["cosine"].to_numpy(np.float32),
        "pruner score": vp["prune_score"].to_numpy(np.float32)}
print("\nPair ranking quality on all validation candidates:")
for name, s in base.items():
    ok = ~np.isnan(s)
    print(f"  {name:22s} AUC {roc_auc_score(VY[ok], s[ok]):.4f} | AP {average_precision_score(VY[ok], s[ok]):.4f}")

pd.DataFrame({"source1_entity_id": vp["source1_entity_id"], "candidate_entity_id": vp["candidate_entity_id"],
              "ce_logit": OOF}).to_parquet(OUT_DIR / f"ce_val_scores{SUFFIX}.parquet", index=False)
print(f"saved ce_val_scores{SUFFIX}.parquet | elapsed {minutes():.0f} min")
del VA, VB, vp; gc.collect()

## Score test candidates
Model 0 scores everything and the result is saved. The next fold model runs only if the projected time still fits
`max_total_minutes`, and the saved file then holds the averaged logits.

In [ ]:
tp = pd.read_parquet(os.path.join(RUN_DIR, "test_candidate_scores.parquet"),
                     columns=["source1_entity_id", "candidate_entity_id"])
if DEBUG_MODE:
    tp = tp.iloc[:CONFIG["debug_test_pairs"]].reset_index(drop=True)
t1 = read_tsv(test_path(DATA_ROOT, 1), ["entity_id", "business_name", "business_address"])
tc = pd.concat([read_tsv(test_path(DATA_ROOT, i), ["entity_id", "business_name", "business_address"]) for i in (2, 3)],
               ignore_index=True)
T1_TEXT = record_text(t1["business_name"], t1["business_address"])
TC_TEXT = record_text(tc["business_name"], tc["business_address"])
qi = pd.Index(t1["entity_id"]).get_indexer(tp["source1_entity_id"])
ci = pd.Index(tc["entity_id"]).get_indexer(tp["candidate_entity_id"])
assert (qi >= 0).all() and (ci >= 0).all(), "candidate ids not found in test files"
del t1, tc; gc.collect()
TA, TB = T1_TEXT[qi], TC_TEXT[ci]
del T1_TEXT, TC_TEXT, qi, ci; gc.collect()
print(f"test pairs {len(tp):,} | elapsed {minutes():.0f} min")

def save_test(logit_sum, n_models):
    pd.DataFrame({"source1_entity_id": tp["source1_entity_id"], "candidate_entity_id": tp["candidate_entity_id"],
                  "ce_logit": (logit_sum / n_models).astype(np.float32)}).to_parquet(
        OUT_DIR / f"ce_test_scores{SUFFIX}.parquet", index=False)
    print(f"saved ce_test_scores{SUFFIX}.parquet (average of {n_models} model(s)) | elapsed {minutes():.0f} min")

logit_sum, n_used, last_dur = np.zeros(len(tp), dtype=np.float64), 0, None
for k, path in enumerate(MODEL_PATHS):
    if last_dur is not None and minutes() + last_dur * 1.1 > CONFIG["max_total_minutes"]:
        print(f"skipping fold model {k}: projected {minutes() + last_dur:.0f} min > budget"); break
    t = time.time()
    model = CrossEncoder(INIT)
    model.load_state_dict({k2: v.float() for k2, v in torch.load(path, map_location="cpu").items()})
    model = model.to(DEV)
    if N_GPU > 1:
        model = nn.DataParallel(model)
    logit_sum += score_pairs(model, TA, TB, desc=f"test model {k}")
    n_used += 1; last_dur = (time.time() - t) / 60
    del model; gc.collect(); torch.cuda.empty_cache()
    save_test(logit_sum, n_used)

print(f"""
==================================================
CROSS-ENCODER SUMMARY
==================================================
init              : {INIT}
validation OOF AUC: {roc_auc_score(VY[~np.isnan(OOF)], OOF[~np.isnan(OOF)]):.4f}
test pairs scored : {len(tp):,} with {n_used} model(s)
outputs           : {OUT_DIR / f'ce_val_scores{SUFFIX}.parquet'}
                    {OUT_DIR / f'ce_test_scores{SUFFIX}.parquet'}
total time        : {minutes():.0f} min
Next: attach this output + the candidate-notebook output to entity_matcher.ipynb.
==================================================""")